In [ ]:
import requests
import time
import json
from pathlib import Path

REFRESH_TOKEN = "MI TOKEN"
ARTIST_ID = 2762

outdir = Path("chartmetric_test")
outdir.mkdir(exist_ok=True)

# 1) token
token_resp = requests.post(
    "https://api.chartmetric.com/api/token",
    json={"refreshtoken": REFRESH_TOKEN}
)
token_resp.raise_for_status()
access_token = token_resp.json()["token"]

headers = {
    "Authorization": f"Bearer {access_token}",
    "X-Accept-Partial-Data": "true"
}

time.sleep(1.1)

# 2) metadata
meta_resp = requests.get(
    f"https://api.chartmetric.com/api/artist/{ARTIST_ID}",
    headers=headers
)
meta_resp.raise_for_status()
meta_json = meta_resp.json()

with open(outdir / f"artist_{ARTIST_ID}_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta_json, f, ensure_ascii=False, indent=2)

time.sleep(1.1)

# 3) past events
events_resp = requests.get(
    f"https://api.chartmetric.com/api/artist/{ARTIST_ID}/past/events",
    headers=headers,
    params={"fromDaysAgo": 365, "toDaysAgo": 0, "limit": 5, "offset": 0}
)
events_resp.raise_for_status()
events_json = events_resp.json()

with open(outdir / f"artist_{ARTIST_ID}_past_events.json", "w", encoding="utf-8") as f:
    json.dump(events_json, f, ensure_ascii=False, indent=2)

print("Archivos guardados:")
print(outdir / f"artist_{ARTIST_ID}_metadata.json")
print(outdir / f"artist_{ARTIST_ID}_past_events.json")

Archivos guardados:
chartmetric_test\artist_2762_metadata.json
chartmetric_test\artist_2762_past_events.json


In [2]:
import json
from pprint import pprint

with open("chartmetric_test/artist_2762_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open("chartmetric_test/artist_2762_past_events.json", "r", encoding="utf-8") as f:
    past_events = json.load(f)

print("=== KEYS METADATA ===")
print(metadata.keys())

print("\n=== KEYS PAST EVENTS ===")
print(past_events.keys())

=== KEYS METADATA ===
dict_keys(['obj'])

=== KEYS PAST EVENTS ===
dict_keys(['obj'])


In [3]:
print("=== TIPO metadata['obj'] ===")
print(type(metadata["obj"]))

print("\n=== TIPO past_events['obj'] ===")
print(type(past_events["obj"]))

print("\n=== PRIMERAS CLAVES DEL OBJETO DE METADATA ===")
print(metadata["obj"].keys())

print("\n=== CANTIDAD DE EVENTOS ===")
print(len(past_events["obj"]))

=== TIPO metadata['obj'] ===
<class 'dict'>

=== TIPO past_events['obj'] ===
<class 'list'>

=== PRIMERAS CLAVES DEL OBJETO DE METADATA ===
dict_keys(['id', 'name', 'created_at', 'code2', 'band', 'pronoun_title', 'gender_title', 'gender', 'isni', 'cm_artist_rank', 'cm_artist_score', 'cover_url', 'image_url', 'hometown_city', 'current_city', 'current_city_id', 'band_members', 'booking_agent', 'record_label', 'press_contact', 'general_manager', 'topSongwriterCollaborators', 'description', 'genres', 'genre_smart_ordered', 'moods', 'activities', 'career_status', 'cm_statistics', 'genreRank', 'subGenreRank1', 'subGenreRank2'])

=== CANTIDAD DE EVENTOS ===
0


In [4]:
print("=== METADATA COMPLETA DEL ARTISTA ===")
pprint(metadata["obj"])

=== METADATA COMPLETA DEL ARTISTA ===
{'activities': [{'id': 504873, 'name': 'daydreaming'},
                {'id': 504872, 'name': 'dancy'},
                {'id': 504877, 'name': 'self-love'},
                {'id': 504871, 'name': 'bonding'},
                {'id': 504887, 'name': 'reassured'},
                {'id': 504876, 'name': 'partying'},
                {'id': 504879, 'name': 'rural'},
                {'id': 504874, 'name': 'driving'}],
 'band': False,
 'band_members': None,
 'booking_agent': None,
 'career_status': {'stage': 'superstar',
                   'stage_score': 89,
                   'trend': 'growth',
                   'trend_score': 69},
 'cm_artist_rank': 1,
 'cm_artist_score': 99.38711679376492,
 'cm_statistics': {'boomplay_comments': None,
                   'boomplay_comments_rank': None,
                   'boomplay_favorites': None,
                   'boomplay_favorites_rank': None,
                   'boomplay_plays': None,
                   'boomplay_

In [6]:
print("=== PRIMER EVENTO ===")
pprint(past_events["obj"])

=== PRIMER EVENTO ===
[]


In [14]:
time.sleep(1.1)

events_resp = requests.get(
    f"https://api.chartmetric.com/api/artist/{ARTIST_ID}/past/events",
    headers=headers,
    params={
        "fromDaysAgo": 60,
        "toDaysAgo": 0,
        "limit": 10,
        "offset": 0
    }
)

print(events_resp.status_code)
print(events_resp.url)

events_resp.raise_for_status()
events_json = events_resp.json()

with open(outdir / f"artist_{ARTIST_ID}_past_events_doc_example.json", "w", encoding="utf-8") as f:
    json.dump(events_json, f, ensure_ascii=False, indent=2)

print("Cantidad de eventos:", len(events_json.get("obj", [])))

200
https://api.chartmetric.com/api/artist/2762/past/events?fromDaysAgo=60&toDaysAgo=0&limit=10&offset=0
Cantidad de eventos: 0


In [16]:
# 4) current events - con parámetros de tiempo
import time
import json
from pprint import pprint

time.sleep(1.1)

current_resp = requests.get(
    f"https://api.chartmetric.com/api/artist/{ARTIST_ID}/current/events",
    headers=headers,
    params={
        "fromDaysAgo": 0,
        "toDaysAgo": -50,
        "limit": 10,
        "offset": 0
    }
)

print("=== STATUS CURRENT ===")
print(current_resp.status_code)

print("\n=== URL CURRENT ===")
print(current_resp.url)

# si falla, mostremos el texto de error antes de cortar
if current_resp.status_code != 200:
    print("\n=== ERROR TEXT ===")
    print(current_resp.text)

current_resp.raise_for_status()
current_json = current_resp.json()

with open(outdir / f"artist_{ARTIST_ID}_current_events.json", "w", encoding="utf-8") as f:
    json.dump(current_json, f, ensure_ascii=False, indent=2)

print("\n=== KEYS DEL JSON CURRENT ===")
print(current_json.keys())

print("\n=== CANTIDAD DE CURRENT EVENTS ===")
print(len(current_json.get("obj", [])))

if len(current_json.get("obj", [])) > 0:
    print("\n=== PRIMER CURRENT EVENT ===")
    pprint(current_json["obj"][0])
else:
    print("\nNo devolvió current events.")

=== STATUS CURRENT ===
200

=== URL CURRENT ===
https://api.chartmetric.com/api/artist/2762/current/events?fromDaysAgo=0&toDaysAgo=-50&limit=10&offset=0

=== KEYS DEL JSON CURRENT ===
dict_keys(['obj'])

=== CANTIDAD DE CURRENT EVENTS ===
0

No devolvió current events.


In [17]:
import requests
import time
import json
from pathlib import Path
from pprint import pprint

# carpeta de salida
outdir = Path("chartmetric_test_multi")
outdir.mkdir(exist_ok=True)

# artistas de prueba tomados de tu head
artists = [
    {"id": 5381, "name": "Dua Lipa"},
    {"id": 214945, "name": "Bad Bunny"},
]

# helper para nombre de archivo seguro
def safe_name(name):
    return (
        name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
    )

results_summary = []

for artist in artists:
    artist_id = artist["id"]
    artist_name = artist["name"]
    slug = safe_name(artist_name)

    print("\n" + "=" * 70)
    print(f"ARTISTA: {artist_name} ({artist_id})")
    print("=" * 70)

    # ---------- METADATA ----------
    time.sleep(1.1)

    meta_resp = requests.get(
        f"https://api.chartmetric.com/api/artist/{artist_id}",
        headers=headers
    )

    print("\n[METADATA]")
    print("status:", meta_resp.status_code)
    print("url:", meta_resp.url)

    if meta_resp.status_code != 200:
        print("error metadata:", meta_resp.text)
        results_summary.append({
            "artist_id": artist_id,
            "artist_name": artist_name,
            "metadata_status": meta_resp.status_code,
            "past_events_status": None,
            "current_events_status": None,
            "n_past_events": None,
            "n_current_events": None,
        })
        continue

    meta_json = meta_resp.json()

    with open(outdir / f"artist_{artist_id}_{slug}_metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta_json, f, ensure_ascii=False, indent=2)

    meta_obj = meta_json.get("obj", {})
    cm_stats = meta_obj.get("cm_statistics", {})

    print("name:", meta_obj.get("name"))
    print("sp_monthly_listeners:", cm_stats.get("sp_monthly_listeners"))
    print("sp_followers:", cm_stats.get("sp_followers"))
    print("ins_followers:", cm_stats.get("ins_followers"))
    print("tiktok_followers:", cm_stats.get("tiktok_followers"))
    print("ycs_subscribers:", cm_stats.get("ycs_subscribers"))

    # ---------- PAST EVENTS ----------
    time.sleep(1.1)

    past_resp = requests.get(
        f"https://api.chartmetric.com/api/artist/{artist_id}/past/events",
        headers=headers,
        params={
            "fromDaysAgo": 365,
            "toDaysAgo": 0,
            "limit": 10,
            "offset": 0
        }
    )

    print("\n[PAST EVENTS]")
    print("status:", past_resp.status_code)
    print("url:", past_resp.url)

    if past_resp.status_code != 200:
        print("error past:", past_resp.text)
        n_past = None
    else:
        past_json = past_resp.json()
        with open(outdir / f"artist_{artist_id}_{slug}_past_events.json", "w", encoding="utf-8") as f:
            json.dump(past_json, f, ensure_ascii=False, indent=2)

        n_past = len(past_json.get("obj", []))
        print("cantidad past events:", n_past)

        if n_past > 0:
            print("keys primer past event:")
            print(past_json["obj"][0].keys())

    # ---------- CURRENT EVENTS ----------
    time.sleep(1.1)

    current_resp = requests.get(
        f"https://api.chartmetric.com/api/artist/{artist_id}/current/events",
        headers=headers,
        params={
            "fromDaysAgo": 0,
            "toDaysAgo": -50,
            "limit": 10,
            "offset": 0
        }
    )

    print("\n[CURRENT EVENTS]")
    print("status:", current_resp.status_code)
    print("url:", current_resp.url)

    if current_resp.status_code != 200:
        print("error current:", current_resp.text)
        n_current = None
    else:
        current_json = current_resp.json()
        with open(outdir / f"artist_{artist_id}_{slug}_current_events.json", "w", encoding="utf-8") as f:
            json.dump(current_json, f, ensure_ascii=False, indent=2)

        n_current = len(current_json.get("obj", []))
        print("cantidad current events:", n_current)

        if n_current > 0:
            print("keys primer current event:")
            print(current_json["obj"][0].keys())

    results_summary.append({
        "artist_id": artist_id,
        "artist_name": artist_name,
        "metadata_status": meta_resp.status_code,
        "past_events_status": past_resp.status_code,
        "current_events_status": current_resp.status_code,
        "n_past_events": n_past,
        "n_current_events": n_current,
    })

print("\n" + "=" * 70)
print("RESUMEN FINAL")
print("=" * 70)
pprint(results_summary)


ARTISTA: Dua Lipa (5381)

[METADATA]
status: 200
url: https://api.chartmetric.com/api/artist/5381
name: Dua Lipa
sp_monthly_listeners: 65015229
sp_followers: 47767361
ins_followers: 88665348
tiktok_followers: 11200000
ycs_subscribers: 24700000

[PAST EVENTS]
status: 200
url: https://api.chartmetric.com/api/artist/5381/past/events?fromDaysAgo=365&toDaysAgo=0&limit=10&offset=0
cantidad past events: 10
keys primer past event:
dict_keys(['id', 'event_name', 'jambase_event_url', 'songkick_event_url', 'seatgeek_event_url', 'ticketmaster_event_url', 'image_url', 'type', 'start_date', 'end_date', 'venue_id', 'venue_name', 'venue_url', 'venue_capacity', 'city', 'code2', 'low_price', 'high_price', 'price', 'price_trend', 'currency', 'is_headliner'])

[CURRENT EVENTS]
status: 200
url: https://api.chartmetric.com/api/artist/5381/current/events?fromDaysAgo=0&toDaysAgo=-50&limit=10&offset=0
cantidad current events: 0

ARTISTA: Bad Bunny (214945)

[METADATA]
status: 200
url: https://api.chartmetric.